# Multi-agent clinical workflows

Exercise notebook for the lesson **[Multi-agent clinical workflows](https://ramkumarmd.com/medical-ai/agentic-ai/multi-agent-clinical-workflows)** in the *Agentic AI* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Multi-agent systems divide a task between several model-driven agents, for example one that gathers facts, one that checks them and one that drafts a summary, sometimes coordinated by an orchestrator. Splitting work can improve focus and allows independent checking, but small error rates compound across steps, and agents built on the same model can share the same blind spots. Clinical multi-agent workflows need independent verification against the source record and a clinician approval gate before anything affects care.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Tool calling against FHIR data](https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling)**: Two synthetic patients, three observations and two medication requests.

In [ ]:
# A small synthetic FHIR R4 bundle (made-up patients, no real data)
bundle = {"resourceType": "Bundle", "entry": [{"resource": r} for r in [
    {"resourceType": "Patient", "id": "p1", "birthDate": "1958-04-02", "gender": "female"},
    {"resourceType": "Patient", "id": "p2", "birthDate": "1971-09-15", "gender": "male"},
    {"resourceType": "Observation", "id": "o1", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-01-10", "valueQuantity": {"value": 8.2, "unit": "%"}},
    {"resourceType": "Observation", "id": "o2", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-07-02", "valueQuantity": {"value": 7.4, "unit": "%"}},
    {"resourceType": "Observation", "id": "o3", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "98979-8", "display": "eGFR (CKD-EPI 2021)"}]},
     "effectiveDateTime": "2026-07-02", "valueQuantity": {"value": 41, "unit": "mL/min/1.73m2"}},
    {"resourceType": "MedicationRequest", "id": "m1", "status": "active", "subject": {"reference": "Patient/p1"},
     "medicationCodeableConcept": {"text": "Metformin 500 mg tablets"}, "authoredOn": "2024-05-01"},
    {"resourceType": "MedicationRequest", "id": "m2", "status": "stopped", "subject": {"reference": "Patient/p1"},
     "medicationCodeableConcept": {"text": "Gliclazide 80 mg tablets"}, "authoredOn": "2023-02-11"},
]]}
resources = [e["resource"] for e in bundle["entry"]]
print({t: sum(r["resourceType"] == t for r in resources)
       for t in ["Patient", "Observation", "MedicationRequest"]})

# {'Patient': 2, 'Observation': 3, 'MedicationRequest': 2}

From **[Tool calling against FHIR data](https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling)**: Continuing from the bundle block: patient-scoped tools with provenance and an audit log.

In [ ]:
class PatientTools:
    """Read-only tools scoped to ONE patient, chosen by the clinician, not by the model."""
    def __init__(self, resources, patient_id):
        self.patient_ref = f"Patient/{patient_id}"
        self.resources = resources
        self.audit = []

    def _mine(self, resource_type):
        return [r for r in self.resources if r["resourceType"] == resource_type
                and r.get("subject", {}).get("reference") == self.patient_ref]

    def latest_observation(self, loinc_code):
        self.audit.append(("latest_observation", loinc_code))
        matches = [o for o in self._mine("Observation")
                   if any(c["code"] == loinc_code for c in o["code"]["coding"])]
        if not matches:
            return {"result": None, "note": f"no observation with LOINC {loinc_code}"}
        o = max(matches, key=lambda o: o["effectiveDateTime"])
        return {"value": o["valueQuantity"]["value"], "unit": o["valueQuantity"]["unit"],
                "date": o["effectiveDateTime"], "source": f"Observation/{o['id']}"}

    def active_medications(self):
        self.audit.append(("active_medications", None))
        return [{"medication": m["medicationCodeableConcept"]["text"],
                 "source": f"MedicationRequest/{m['id']}"}
                for m in self._mine("MedicationRequest") if m["status"] == "active"]

tools = PatientTools(resources, patient_id="p1")
print(tools.latest_observation("4548-4"))
print(tools.latest_observation("98979-8"))
print(tools.active_medications())
print(tools.latest_observation("2093-3"))
print("audit log:", tools.audit)

# {'value': 7.4, 'unit': '%', 'date': '2026-07-02', 'source': 'Observation/o2'}
# {'value': 41, 'unit': 'mL/min/1.73m2', 'date': '2026-07-02', 'source': 'Observation/o3'}
# [{'medication': 'Metformin 500 mg tablets', 'source': 'MedicationRequest/m1'}]
# {'result': None, 'note': 'no observation with LOINC 2093-3'}
# audit log: [('latest_observation', '4548-4'), ('latest_observation', '98979-8'), ('active_medications', None), ('latest_observation', '2093-3')]

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Errors compound

End-to-end success for chains of 1 to 10 steps at three per-step success rates.

In [ ]:
for per_step in [0.99, 0.95, 0.90]:
    print(f"per-step success {per_step:.0%}: " + ", ".join(
        f"{n} steps {per_step ** n:.0%}" for n in [1, 3, 5, 10]))

# per-step success 99%: 1 steps 99%, 3 steps 97%, 5 steps 95%, 10 steps 90%
# per-step success 95%: 1 steps 95%, 3 steps 86%, 5 steps 77%, 10 steps 60%
# per-step success 90%: 1 steps 90%, 3 steps 73%, 5 steps 59%, 10 steps 35%

### A pipeline with an independent checker

Run the bundle and PatientTools blocks from the FHIR lesson first.

In [ ]:
tools = PatientTools(resources, patient_id="p1")

def extractor_agent(tools):
    """Gathers facts. Simulated mistake: reports the OLDER HbA1c (8.2%) instead of the latest."""
    return {"hba1c": {"value": 8.2, "source": "Observation/o1"},
            "egfr": tools.latest_observation("98979-8"),
            "medications": tools.active_medications()}

def checker_agent(facts, tools):
    """Independently re-queries the record and compares; flags any mismatch."""
    issues = []
    latest = tools.latest_observation("4548-4")
    if facts["hba1c"]["value"] != latest["value"]:
        issues.append(f"HbA1c {facts['hba1c']['value']} ({facts['hba1c']['source']}) is not the latest: "
                      f"{latest['value']} on {latest['date']} ({latest['source']})")
    on_metformin = any("metformin" in m["medication"].lower() for m in facts["medications"])
    if on_metformin and facts["egfr"]["value"] < 45:
        issues.append(f"On metformin with eGFR {facts['egfr']['value']} (below 45): review with prescriber")
    return issues

def drafter_agent(facts, issues):
    status = "NEEDS CLINICIAN REVIEW" if issues else "draft"
    lines = [f"[{status}] Diabetes review summary (synthetic patient p1)"]
    lines += [f"- HbA1c {facts['hba1c']['value']}%", f"- eGFR {facts['egfr']['value']}"]
    lines += [f"- FLAG: {i}" for i in issues]
    return "\n".join(lines)

facts = extractor_agent(tools)
issues = checker_agent(facts, tools)
print(drafter_agent(facts, issues))

# [NEEDS CLINICIAN REVIEW] Diabetes review summary (synthetic patient p1)
# - HbA1c 8.2%
# - eGFR 41
# - FLAG: HbA1c 8.2 (Observation/o1) is not the latest: 7.4 on 2026-07-02 (Observation/o2)
# - FLAG: On metformin with eGFR 41 (below 45): review with prescriber

## Exercise

**Goal:** Build a small multi-agent pipeline over synthetic FHIR data, inject errors and measure how well independent checks catch them.

**Data:** Synthetic FHIR bundle from the previous lesson (Generated in the notebook)

### Step 1

Reproduce the extractor, checker and drafter pipeline.

In [ ]:
# Your code here


### Step 2

Inject five different extractor errors (wrong value, stopped medicine reported as active, wrong unit, missing allergy, wrong patient) and record which the checker catches.

In [ ]:
# Your code here


### Step 3

Extend the checker to catch the errors it missed, using only re-queries of the source record.

In [ ]:
# Your code here


### Step 4

Simulate 1,000 runs with random step failures (95% per step, 4 steps) with and without the checker, and compare end-to-end error rates.

In [ ]:
# Your code here


### Step 5

Write the approval rules you would propose for a pre-clinic summary agent in your setting.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/agentic-ai/multi-agent-clinical-workflows](https://ramkumarmd.com/medical-ai/agentic-ai/multi-agent-clinical-workflows). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.